In [1]:
import os
for _var in ("VECLIB_MAXIMUM_THREADS", "OPENBLAS_NUM_THREADS",
             "OMP_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[_var] = "1"

import numpy as np

np.set_printoptions(precision=4, suppress=True)
print("BLAS threading pinned to 1 before import: single-task-per-core is the measured regime")

# --- gamma matrices, Dirac-Pauli basis --------------------------------------
s1 = np.array([[0, 1], [1, 0]], dtype=complex)   # sigma_x
s2 = np.array([[0, -1j], [1j, 0]], dtype=complex)  # sigma_y
s3 = np.array([[1, 0], [0, -1]], dtype=complex)   # sigma_z
I2 = np.eye(2, dtype=complex)

sigma = [s1, s2, s3]
g0 = np.diag([1, 1, -1, -1]).astype(complex)
g = [g0] + [np.block([[np.zeros((2, 2)), s], [-s, np.zeros((2, 2))]])
            for s in sigma]

eta = np.diag([1, -1, -1, -1]).astype(complex)

dev = 0.0
for mu in range(4):
    for nu in range(4):
        anti = g[mu] @ g[nu] + g[nu] @ g[mu]
        rhs = (2.0 * eta[mu, nu]) * np.eye(4, dtype=complex)
        dev = max(dev, float(np.max(np.abs(anti - rhs))))
print("Clifford {gm,gn} = 2 eta^{mn} 1  :  max deviation =", f"{dev:.2e}")

g5 = 1j * g[0] @ g[1] @ g[2] @ g[3]
print("g5^2 == 1                     :", float(np.max(np.abs(g5 @ g5 - np.eye(4)))) == 0.0)
anti5 = max(float(np.max(np.abs(g5 @ g[mu] + g[mu] @ g5))) for mu in range(4))
print("g5 anticommutes with all gm   :", f"{anti5:.2e}")


BLAS threading pinned to 1 before import: single-task-per-core is the measured regime
Clifford {gm,gn} = 2 eta^{mn} 1  :  max deviation = 0.00e+00
g5^2 == 1                     : True
g5 anticommutes with all gm   : 0.00e+00


In [2]:
m = 1.0
pz = 1.4
E = float(np.sqrt(pz ** 2 + m ** 2))
s_dot_p = pz * s3                      # momentum along z

spinors = {}
for sidx, chi in enumerate((np.array([1, 0], dtype=complex),
                            np.array([0, 1], dtype=complex))):
    top = np.sqrt(E + m) * chi
    bot = ((s_dot_p / np.sqrt(E + m)) @ chi).reshape(2)
    u = np.concatenate([top, bot])
    spinors[sidx] = u

pslash = g0 * E - g[3] * pz            # gamma^0 E - gamma^3 pz
print(f"E = {E:.6f}   (pz = {pz}, m = {m})")
print(f"{'s':>2} {'u+u  = 2E':>18} {'ubar u = 2m':>14} {'|(pslash-m)u|':>16}")
for sidx, u in spinors.items():
    u_dag_u = float(np.real(u.conj() @ u))
    ubar = u.conj() @ g0
    ubar_u = float(np.real(ubar @ u))
    resid = float(np.linalg.norm(pslash @ u - m * u))
    print(f"{sidx:>2} {u_dag_u:>18.10f} {ubar_u:>14.10f} {resid:>16.2e}")
print(f"2E = {2*E:.10f}    2m = {2*m:.10f}")


E = 1.720465   (pz = 1.4, m = 1.0)
 s          u+u  = 2E    ubar u = 2m    |(pslash-m)u|
 0       3.4409301068   2.0000000000         2.22e-16
 1       3.4409301068   2.0000000000         2.22e-16
2E = 3.4409301068    2m = 2.0000000000


In [3]:
E, m = 2.0, 1.0
k = float(np.sqrt(E ** 2 - m ** 2))
H2d = np.block([[m * np.eye(1), np.array([[k]])],
                [np.array([[k]]), -m * np.eye(1)]])   # alpha*k + beta*m
vals, vecs = np.linalg.eigh(H2d)                       # symmetric: exact
ratio_num = float((vecs[:, 1][1] / vecs[:, 1][0]).real)   # E = +2 branch
ratio_exact = k / (E + m)
print(f"E = {E},  m = {m},  k = {k:.9f}   [sqrt(3) = {np.sqrt(3):.9f}]")
print(f"eigensystem eigenvalues (order +sqrt(k^2+m^2)) : {vals}  -> +- 2 as required")
print(f"eigenspinor ratio (E=+2 branch) : {ratio_num:.15f}")
print(f"analytic ratio   k/(E+m)        : {ratio_exact:.15f}")
print(f"deviation                       : {abs(ratio_num - ratio_exact):.2e}")

# exact real-space solution residual on a dense grid
x = np.linspace(0.0, 6.0, 2001)
u = np.cos(k * x)
v = -(k / (E + m)) * np.sin(k * x)
du = np.gradient(u, x)                       # FD derivative is O(h^2)
dv = np.gradient(v, x)
r1 = du - (E + m) * v                        # du/dx = +(E+m) v
r2 = dv + (E - m) * u                        # dv/dx = -(E-m) u
print(f"closed-form solution residual (O(h^2) FD derivative): "
      f"max|r1| = {np.max(np.abs(r1)):.2e},  max|r2| = {np.max(np.abs(r2)):.2e}")


E = 2.0,  m = 1.0,  k = 1.732050808   [sqrt(3) = 1.732050808]
eigensystem eigenvalues (order +sqrt(k^2+m^2)) : [-2.  2.]  -> +- 2 as required
eigenspinor ratio (E=+2 branch) : 0.577350269189626
analytic ratio   k/(E+m)        : 0.577350269189626
deviation                       : 0.00e+00
closed-form solution residual (O(h^2) FD derivative): max|r1| = 4.50e-03,  max|r2| = 2.14e-03


In [4]:
L, N = 12.0, 512
h = L / N
D = np.zeros((N, N))
for j in range(N):
    D[j, (j + 1) % N] = 1.0
    D[j, (j - 1) % N] = -1.0
D /= (2.0 * h)                               # antisymmetric central difference
neg_iD = -1j * D                             # exactly Hermitian
I_grid = np.eye(N)
# H = alpha (-i d/dx) + beta m in the (u, v) basis:
H = np.zeros((2 * N, 2 * N), dtype=complex)
H[:N, :N] = m * I_grid                       # beta = sigma_z: +m on the u block
H[N:, N:] = -m * I_grid                      #                  -m on the v block
H[:N, N:] = neg_iD                           # alpha = sigma_x off-diagonal
H[N:, :N] = neg_iD
vals_sorted = np.sort(np.linalg.eigvalsh(H))

# exact DISCRETE dispersion: eigenvalue of -iD on mode j is sin(theta_j)/h
theta = 2.0 * np.pi * np.arange(N) / N
kdisc = np.sin(theta) / h
Etheory = np.sort(np.concatenate([np.sqrt(kdisc ** 2 + m ** 2),
                                  -np.sqrt(kdisc ** 2 + m ** 2)]))
dev = float(np.max(np.abs(vals_sorted - Etheory)))
print(f"grid N={N}, L={L}, h={h:.4f}: FD spectrum vs discrete dispersion +-sqrt(k~^2+m^2)")
print(f"max |spectrum discrepancy| = {dev:.2e}   (round-off, as it must be)")
print(f"lowest eigenvalue (Dirac sea)  : {vals_sorted[0]:.9f}")
print(f"highest eigenvalue             : {vals_sorted[-1]:.9f}")
print(f"pair symmetry max |E_j + E_-j| : "
      f"{np.max(np.abs(vals_sorted[:N] + vals_sorted[N:][::-1])):.2e}")


grid N=512, L=12.0, h=0.0234: FD spectrum vs discrete dispersion +-sqrt(k~^2+m^2)
max |spectrum discrepancy| = 1.63e-13   (round-off, as it must be)
lowest eigenvalue (Dirac sea)  : -42.678383808
highest eigenvalue             : 42.678383808
pair symmetry max |E_j + E_-j| : 1.49e-13
